In [6]:
import os
import re
import unicodedata

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt


In [7]:
import numpy as np
import networkx as nx

## 选民分布图

In [8]:
# ========== 0) 路径 ==========
gpkg_path = r"D:\桌面\研一\DATA70202 Applying Data Science 202526 2nd Semester\Data\Continente_CAOP2024_1.gpkg"
out_dir = os.path.join(os.path.dirname(gpkg_path), "outputs_lisboa_porto")
os.makedirs(out_dir, exist_ok=True)

In [9]:
# ========== 1) 读取 municipios 图层 ==========
muni = gpd.read_file(gpkg_path, layer="cont_municipios")
print("Columns:", list(muni.columns))
print("CRS:", muni.crs)
print(muni.head(2))

Columns: ['dtmn', 'municipio', 'distrito_ilha', 'nuts3', 'nuts2', 'nuts1', 'area_ha', 'perimetro_km', 'n_freguesias', 'geometry']
CRS: EPSG:3763
   dtmn           municipio distrito_ilha             nuts3   nuts2  \
0  0101              Águeda        Aveiro  Região de Aveiro  Centro   
1  0102  Albergaria-a-Velha        Aveiro  Região de Aveiro  Centro   

        nuts1   area_ha  perimetro_km  n_freguesias  \
0  Continente  33527.44           105            15   
1  Continente  15882.50            82             6   

                                            geometry  
0  MULTIPOLYGON (((-25647.507 93242.339, -25739.7...  
1  MULTIPOLYGON (((-32903.475 104565.493, -32989....  


In [10]:
# 你这个图层的真实字段
distr_col = "distrito_ilha"
muni_col  = "municipio"

print("Rows:", len(muni))
print("Unique distritos:", sorted(muni[distr_col].unique())[:30])


# ====== 文本规范化：去重音 + 小写 + 去多余空格 ======
def norm_txt(x: str) -> str:
    if x is None:
        return ""
    x = str(x).strip()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(ch for ch in x if not unicodedata.combining(ch))  # 去重音
    x = re.sub(r"\s+", " ", x)
    return x.lower()

muni["distrito_n"] = muni[distr_col].map(norm_txt)
muni["municipio_n"] = muni[muni_col].map(norm_txt)


# ====== 选民数据（你给的 2024 注册选民数） ======
raw = [
    ("Lisboa", "Alenquer", 37091),
    ("Lisboa", "Arruda dos Vinhos", 11861),
    ("Lisboa", "Azambuja", 17691),
    ("Lisboa", "Cadaval", 12273),
    ("Lisboa", "Cascais", 177628),
    ("Lisboa", "Lisboa", 465439),
    ("Lisboa", "Loures", 168472),
    ("Lisboa", "Lourinhã", 23746),
    ("Lisboa", "Mafra", 69811),
    ("Lisboa", "Oeiras", 145353),
    ("Lisboa", "Sintra", 323643),
    ("Lisboa", "Sobral de Monte Agraço", 8891),
    ("Lisboa", "Torres Vedras", 70586),
    ("Lisboa", "Vila Franca de Xira", 114480),
    ("Lisboa", "Amadora", 142381),
    ("Lisboa", "Odivelas", 125826),

    ("Porto", "Amarante", 49980),
    ("Porto", "Baião", 16295),
    ("Porto", "Felgueiras", 51707),
    ("Porto", "Gondomar", 145883),
    ("Porto", "Lousada", 42199),
    ("Porto", "Maia", 117071),
    ("Porto", "Marco de Canaveses", 45450),
    ("Porto", "Matosinhos", 150490),
    ("Porto", "Paços de Ferreira", 49291),
    ("Porto", "Paredes", 75337),
    ("Porto", "Penafiel", 62239),
    ("Porto", "Porto", 204547),
    ("Porto", "Póvoa de Varzim", 60733),
    ("Porto", "Santo Tirso", 62012),
    ("Porto", "Valongo", 84679),
    ("Porto", "Vila do Conde", 72197),
    ("Porto", "Vila Nova de Gaia", 267659),
    ("Porto", "Trofa", 33990),
]
voters = pd.DataFrame(raw, columns=["distrito", "municipio", "voters"])
voters["distrito_n"] = voters["distrito"].map(norm_txt)
voters["municipio_n"] = voters["municipio"].map(norm_txt)


# ====== 过滤 Lisboa / Porto，并 join ======
muni_lp = muni[muni["distrito_n"].isin([norm_txt("Lisboa"), norm_txt("Porto")])].copy()

muni_lp = muni_lp.merge(
    voters[["distrito_n", "municipio_n", "voters"]],
    on=["distrito_n", "municipio_n"],
    how="left"
)

# 检查匹配情况
missing = muni_lp[muni_lp["voters"].isna()][[distr_col, muni_col]].drop_duplicates()
if len(missing) > 0:
    print("\n[WARNING] 这些 municipios 没有匹配到选民数据（需要检查拼写/是否是同名异写）：")
    print(missing.to_string(index=False))
else:
    print("\nAll municipios matched voters data ✅")


# ====== 分成两个 GeoDataFrame ======
lis_gdf = muni_lp[muni_lp["distrito_n"] == norm_txt("lisboa")].copy()
por_gdf = muni_lp[muni_lp["distrito_n"] == norm_txt("porto")].copy()

print("\nLisboa municipios:", len(lis_gdf), " | Porto municipios:", len(por_gdf))


# ====== 导出两个文件 ======
lis_gpkg = os.path.join(out_dir, "Lisboa_municipios_with_voters.gpkg")
por_gpkg = os.path.join(out_dir, "Porto_municipios_with_voters.gpkg")
lis_gdf.to_file(lis_gpkg, layer="municipios", driver="GPKG")
por_gdf.to_file(por_gpkg, layer="municipios", driver="GPKG")

# GeoJSON（可选）
lis_geojson = os.path.join(out_dir, "Lisboa_municipios_with_voters.geojson")
por_geojson = os.path.join(out_dir, "Porto_municipios_with_voters.geojson")
lis_gdf.to_file(lis_geojson, driver="GeoJSON")
por_gdf.to_file(por_geojson, driver="GeoJSON")

print("\nSaved to:", out_dir)


# ====== 画 choropleth 地图 ======
def plot_choropleth(gdf, title, out_png):
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    gdf.plot(
        column="voters",
        ax=ax,
        legend=True,
        missing_kwds={"color": "lightgrey", "label": "No data"},
        edgecolor="black",
        linewidth=0.4
    )
    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close(fig)

Rows: 278
Unique distritos: ['Aveiro', 'Beja', 'Braga', 'Bragança', 'Castelo Branco', 'Coimbra', 'Faro', 'Guarda', 'Leiria', 'Lisboa', 'Portalegre', 'Porto', 'Santarém', 'Setúbal', 'Viana do Castelo', 'Vila Real', 'Viseu', 'Évora']

All municipios matched voters data ✅

Lisboa municipios: 16  | Porto municipios: 18

Saved to: D:\桌面\研一\DATA70202 Applying Data Science 202526 2nd Semester\Data\outputs_lisboa_porto


In [11]:
def plot_choropleth_with_labels(gdf, title, out_png, name_col="municipio", value_col="voters"):
    fig, ax = plt.subplots(1, 1, figsize=(9, 9))

    # 1) 先画底图（颜色分布）
    gdf.plot(
        column=value_col,
        ax=ax,
        legend=True,
        missing_kwds={"color": "lightgrey", "label": "No data"},
        edgecolor="black",
        linewidth=0.4
    )

    # 2) 生成“落在多边形内部”的标注点（比 centroid 更稳）
    # 注意：representative_point 一定在面内部
    pts = gdf.geometry.representative_point()

    # 3) 加标签：Municipio + voters（你也可以只标 voters，见下方注释）
    for (x, y), (_, row) in zip(zip(pts.x, pts.y), gdf.iterrows()):
        if pd.isna(row[value_col]):
            continue
        label = f"{row[name_col]}\n{int(row[value_col]):,}"   # 逗号分隔
        ax.annotate(
            label,
            xy=(x, y),
            ha="center", va="center",
            fontsize=7,
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7)
        )

        # 如果你只想显示数字，把上面 label 改成：
        # label = f"{int(row[value_col]):,}"

    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()

    # 4) 保存：同名覆盖旧图（你要求“新的图覆盖旧图”）
    plt.savefig(out_png, dpi=300)
    plt.close(fig)

In [12]:
plot_choropleth_with_labels(
    lis_gdf, "Lisboa (Municipios) - Registered Voters 2024",
    os.path.join(out_dir, "Lisboa_voters_choropleth.png"),
    name_col="municipio", value_col="voters"
)

plot_choropleth_with_labels(
    por_gdf, "Porto (Municipios) - Registered Voters 2024",
    os.path.join(out_dir, "Porto_voters_choropleth.png"),
    name_col="municipio", value_col="voters"
)

# 自动拆分里斯本和波尔图

In [37]:
from libpysal import weights
import networkx as nx
import numpy as np

def build_adjacency_graph(gdf):
    # 使用 Queen 邻接构建图
    w = weights.Queen.from_dataframe(gdf)
    return w.to_networkx()

def contiguous_grow_partition(gdf, G, k, value_col="voters", random_state=0):
    """严格连续的初始分区生长算法"""
    np.random.seed(random_state)
    assign = np.full(len(gdf), -1, dtype=int)
    
    # 1. 随机选种子，但种子之间最好有一定距离（简单处理：直接随机）
    seeds = np.random.choice(list(G.nodes), size=k, replace=False)
    for i, s in enumerate(seeds):
        assign[s] = i
    
    # 2. 队列式扩张，确保每一步都连通
    unassigned = set(G.nodes) - set(seeds)
    while unassigned:
        changed = False
        # 遍历所有已分配选区的边缘节点
        for u in np.random.permutation(list(unassigned)):
            # 找到 u 的邻居中已经分配了选区的节点
            neighbor_assigns = [assign[v] for v in G.neighbors(u) if assign[v] != -1]
            if neighbor_assigns:
                # 策略：分给那个目前总票数最少的邻居选区
                temp_sums = {i: gdf.iloc[assign == i][value_col].sum() for i in set(neighbor_assigns)}
                best_k = min(temp_sums, key=temp_sums.get)
                assign[u] = best_k
                unassigned.remove(u)
                changed = True
                break 
        if not changed: break # 防止孤岛
    return assign



In [38]:
def local_balance_improve(gdf, G, assign, k, value_col="voters", iters=10000):
    """
    改进版：以目标选民数偏差最小化为目标的局部搜索算法
    """
    new_assign = assign.copy()
    vals = gdf[value_col].values
    total_voters = vals.sum()
    # 对应图片中的 E_target
    e_target = total_voters / k 
    
    def get_score(current_assign):
        # 计算每个选区与目标的绝对偏差之和
        current_sums = [vals[current_assign == i].sum() for i in range(k)]
        return sum(abs(s - e_target) for s in current_sums)

    current_score = get_score(new_assign)

    for i in range(iters):
        # 随机选择一个边界节点
        u = np.random.choice(list(G.nodes))
        curr_k = new_assign[u]
        
        neighbors = list(G.neighbors(u))
        other_ks = list(set(new_assign[v] for v in neighbors if new_assign[v] != curr_k))
        
        if not other_ks:
            continue
            
        target_k = np.random.choice(other_ks)
        
        # 连通性检查：确保 curr_k 失去 u 后依然连通
        nodes_in_curr = [n for n in G.nodes if new_assign[n] == curr_k and n != u]
        if nodes_in_curr:
            if not nx.is_connected(G.subgraph(nodes_in_curr)):
                continue
        
        # 尝试移动并评分
        old_k = new_assign[u]
        new_assign[u] = target_k
        new_score = get_score(new_assign)
        
        # 如果新分值更低（更接近 E_target），则接受移动
        if new_score < current_score:
            current_score = new_score
        else:
            # 否则回退
            new_assign[u] = old_k
            
        # 每 1000 次尝试打印一次当前偏差情况（可选）
        if i % 2000 == 0:
            current_sums = [vals[new_assign == j].sum() for j in range(k)]
            devs = [abs(s - e_target) / e_target for s in current_sums]
            print(f"Iter {i}: Max Deviation = {max(devs):.2%}")

    return new_assign

In [29]:
def split_into_four(gdf, title_prefix, k=4, value_col="voters", name_col="municipio",
                    random_state=0, improve_iters=2000):
    gdf = gdf.reset_index(drop=True).copy()

    # 1. 构建邻接图
    G = build_adjacency_graph(gdf)

    if not nx.is_connected(G):
        comps = nx.number_connected_components(G)
        print(f"[WARNING] {title_prefix}: adjacency graph not connected ({comps} components).")

    # 2. 初始连续增长分区 (注意：这里去掉了 target 参数)
    assign = contiguous_grow_partition(
        gdf, G, k=k, value_col=value_col, random_state=random_state
    )

    # 3. 局部平衡优化 (带连通性检查)
    assign2 = local_balance_improve(
        gdf, G, assign, k=k, value_col=value_col, iters=improve_iters
    )

    gdf["new_sd"] = assign2 + 1  # 编号转为 1..k

    # 4. 汇总数据
    summary = (gdf.groupby("new_sd")[value_col]
               .sum()
               .reset_index()
               .sort_values("new_sd"))
    summary["share"] = summary[value_col] / summary[value_col].sum()

    # 5. 分配表
    assign_table = (gdf[[name_col, value_col, "new_sd"]]
                    .sort_values(["new_sd", value_col], ascending=[True, False])
                    .reset_index(drop=True))

    print(f"\n== {title_prefix} new {k} districts summary ==")
    print(summary.to_string(index=False))

    return gdf, summary, assign_table

In [39]:
import pandas as pd

# 配置参数
combined_excel_path = os.path.join(out_dir, "Balanced_Districts_Report.xlsx")

# 1. 运行 Lisboa (k=3)
# 增加迭代次数以寻求更优解
lis_split, lis_summary, lis_assign = split_into_four(
    lis_gdf, "Lisboa", k=3, random_state=42, improve_iters=10000
)

# 2. 运行 Porto (k=2)
por_split, por_summary, por_assign = split_into_four(
    por_gdf, "Porto", k=2, random_state=123, improve_iters=10000
)

# 3. 计算并添加偏差评估字段（验证是否符合 |E_sub - E_target| <= delta）
for summ, k_val in [(lis_summary, 3), (por_summary, 2)]:
    target = summ["voters"].sum() / k_val
    summ["target_val"] = target
    summ["deviation"] = (summ["voters"] - target) / target
    summ["meets_5pct_std"] = summ["deviation"].abs() <= 0.05

# 4. 集合保存到同一个 Excel
with pd.ExcelWriter(combined_excel_path, engine="openpyxl") as writer:
    lis_summary.to_excel(writer, sheet_name="Lisboa_Summary", index=False)
    lis_assign.to_excel(writer, sheet_name="Lisboa_Details", index=False)
    por_summary.to_excel(writer, sheet_name="Porto_Summary", index=False)
    por_assign.to_excel(writer, sheet_name="Porto_Details", index=False)

print(f"\n✅ 优化后的结果已保存至: {combined_excel_path}")

C:\Users\89364\AppData\Local\Temp\ipykernel_18456\2036385345.py:7: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = weights.Queen.from_dataframe(gdf)


Iter 2000: Max Deviation = 4.32%
Iter 4000: Max Deviation = 4.32%

== Lisboa new 3 districts summary ==
 new_sd  voters    share
      1  660728 0.344997
      2  643652 0.336081
      3  610792 0.318923


C:\Users\89364\AppData\Local\Temp\ipykernel_18456\2036385345.py:7: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = weights.Queen.from_dataframe(gdf)


Iter 6000: Max Deviation = 0.79%
Iter 8000: Max Deviation = 0.79%

== Porto new 2 districts summary ==
 new_sd  voters    share
      1  802143 0.503935
      2  789616 0.496065

✅ 优化后的结果已保存至: D:\桌面\研一\DATA70202 Applying Data Science 202526 2nd Semester\Data\outputs_lisboa_porto\Balanced_Districts_Report.xlsx


In [42]:
def plot_voters_with_newdistrict_outline(
    gdf,
    title,
    out_png,
    voters_col="voters",
    muni_name_col="municipio",
    new_sd_col="new_sd",
    label_fontsize=6,
    outline_lw=2.8,
    outline_color="#EC5F7D",   # 新选区边界统一颜色（避免重复叠色造成碎段）
    show_sd_labels=True            # 是否在每个新区中心标 SD 编号
):
    """
    底图：按 voters_col 渐变着色
    叠加：把 4 个新区的边界合并后只画一次（修复“看起来很多个”）
    标签：每个 municipio 标注 voters
    底部：列出每个新选区总 voters
    """

    gdf = gdf.copy()

    # ---- 1) 计算每个新区的总选民数 ----
    summary = (gdf.groupby(new_sd_col)[voters_col]
               .sum()
               .sort_index()
               .reset_index())
    total = summary[voters_col].sum()
    summary["pct"] = summary[voters_col] / total

    footer = " | ".join([
        f"SD{int(r[new_sd_col])}: {int(r[voters_col]):,} ({r['pct']:.1%})"
        for _, r in summary.iterrows()
    ])
    footer = "New districts totals — " + footer

    # ---- 2) dissolve 得到新区面 ----
    dissolved = gdf.dissolve(by=new_sd_col, as_index=False)

    # ✅ 关键修复：合并所有新区边界，只画一次（共享边界不会被重复画）
    boundary_union = dissolved.geometry.boundary.unary_union

    # ---- 3) 画图：上面地图 + 下面文字区域 ----
    fig = plt.figure(figsize=(10, 11))
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[15, 1])
    ax = fig.add_subplot(gs[0, 0])
    ax_footer = fig.add_subplot(gs[1, 0])

    # 底图：选民数分布
    gdf.plot(
        column=voters_col,
        ax=ax,
        legend=True,
        edgecolor="black",
        linewidth=0.4,
        missing_kwds={"color": "lightgrey", "label": "No data"},
    )

    # 叠加：新区边界（只画一次、统一颜色）
    gpd.GeoSeries([boundary_union], crs=gdf.crs).plot(
        ax=ax,
        color=outline_color,
        linewidth=outline_lw
    )

    # （可选）在新区中心标 SD1~SD4
    if show_sd_labels:
        sd_pts = dissolved.geometry.representative_point()
        for (x, y), sd in zip(zip(sd_pts.x, sd_pts.y), dissolved[new_sd_col].tolist()):
            ax.text(
                x, y, f"SD{int(sd)}",
                ha="center", va="center",
                fontsize=12,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85)
            )

    # 标签：每个 municipio 的选民数
    pts = gdf.geometry.representative_point()
    for (x, y), (_, row) in zip(zip(pts.x, pts.y), gdf.iterrows()):
        if pd.isna(row[voters_col]):
            continue
        ax.annotate(
            text=f"{int(row[voters_col]):,}",
            xy=(x, y),
            ha="center", va="center",
            fontsize=label_fontsize,
            bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.75)
        )

    ax.set_title(title)
    ax.axis("off")

    # 底部汇总文字
    ax_footer.axis("off")
    ax_footer.text(
        0.5, 0.5, footer,
        ha="center", va="center",
        fontsize=10
    )

    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close(fig)

    return summary

In [43]:
# 1. 运行算法生成数据
lis_split, lis_summary, lis_assign = split_into_four(
    lis_gdf, "Lisboa", k=3, random_state=42, improve_iters=5000
)

por_split, por_summary, por_assign = split_into_four(
    por_gdf, "Porto", k=2, random_state=42, improve_iters=5000
)

# 2. 集合保存到同一个 Excel
combined_excel_path = os.path.join(out_dir, "Final_Partition_Report.xlsx")

with pd.ExcelWriter(combined_excel_path, engine="openpyxl") as writer:
    # 写入 Lisboa
    lis_summary.to_excel(writer, sheet_name="Lisboa_Summary", index=False)
    lis_assign.to_excel(writer, sheet_name="Lisboa_Details", index=False)
    
    # 写入 Porto
    por_summary.to_excel(writer, sheet_name="Porto_Summary", index=False)
    por_assign.to_excel(writer, sheet_name="Porto_Details", index=False)

# 3. 同时也生成图片（保持之前的绘图逻辑）
plot_voters_with_newdistrict_outline(
    lis_split, "Lisboa — 3 New Districts by Algorithm", 
    os.path.join(out_dir, "Lisboa_voters_split3_al.png"), 
    voters_col="voters", new_sd_col="new_sd"
)

plot_voters_with_newdistrict_outline(
    por_split, "Porto — 2 New Districts by Algorithm", 
    os.path.join(out_dir, "Porto_voters_split2_al.png"), 
    voters_col="voters", new_sd_col="new_sd"
)

C:\Users\89364\AppData\Local\Temp\ipykernel_18456\2036385345.py:7: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = weights.Queen.from_dataframe(gdf)


Iter 2000: Max Deviation = 4.32%
Iter 4000: Max Deviation = 4.32%

== Lisboa new 3 districts summary ==
 new_sd  voters    share
      1  660728 0.344997
      2  643652 0.336081
      3  610792 0.318923


C:\Users\89364\AppData\Local\Temp\ipykernel_18456\2036385345.py:7: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = weights.Queen.from_dataframe(gdf)



== Porto new 2 districts summary ==
 new_sd  voters    share
      1  789616 0.496065
      2  802143 0.503935


C:\Users\89364\AppData\Local\Temp\ipykernel_18456\611530536.py:40: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  boundary_union = dissolved.geometry.boundary.unary_union
C:\Users\89364\AppData\Local\Temp\ipykernel_18456\611530536.py:40: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  boundary_union = dissolved.geometry.boundary.unary_union


,new_sd,voters,pct
0,1,789616,0.496065
1,2,802143,0.503935


# 生成手动分法的地图

In [47]:
import pandas as pd
import os

# ====== 1) 手动定义映射字典 (规范化名称) ======
manual_mapping = {
    # LISBOA
    "amadora": 1, "lisboa": 1,
    "cascais": 2, "oeiras": 2, "sintra": 2,
    "alenquer": 3, "arruda dos vinhos": 3, "azambuja": 3, "cadaval": 3, 
    "loures": 3, "lourinha": 3, "mafra": 3, "odivelas": 3, 
    "sobral de monte agraco": 3, "torres vedras": 3, "vila franca de xira": 3,

    # PORTO
    "matosinhos": 1, "gondomar": 1, "porto": 1, "vila nova de gaia": 1,
    "povoa de varzim": 2, "vila do conde": 2, "maia": 2, "trofa": 2, 
    "valongo": 2, "santo tirso": 2, "pacos de ferreira": 2, "lousada": 2,
    "paredes": 2, "penafiel": 2, "marco de canaveses": 2, "baiao": 2, 
    "amarante": 2, "felgueiras": 2
}

# ====== 2) 应用分配并生成结果 ======

# 处理 Lisboa
lis_manual = lis_gdf.copy()
lis_manual["new_sd"] = lis_manual["municipio_n"].map(manual_mapping)

# 处理 Porto
por_manual = por_gdf.copy()
por_manual["new_sd"] = por_manual["municipio_n"].map(manual_mapping)

# ====== 3) 生成图片 ======

# 保存 Lisboa 手动分配图
plot_voters_with_newdistrict_outline(
    lis_manual,
    title="Lisboa — 3 New Districts by Manual",
    out_png=os.path.join(out_dir, "Lisboa_voters_split3_ml.png"),
    voters_col="voters",
    new_sd_col="new_sd"
)

# 保存 Porto 手动分配图
plot_voters_with_newdistrict_outline(
    por_manual,
    title="Porto — 2 New Districts by Manual",
    out_png=os.path.join(out_dir, "Porto_voters_split2_ml.png"),
    voters_col="voters",
    new_sd_col="new_sd"
)

C:\Users\89364\AppData\Local\Temp\ipykernel_18456\611530536.py:40: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  boundary_union = dissolved.geometry.boundary.unary_union
C:\Users\89364\AppData\Local\Temp\ipykernel_18456\611530536.py:40: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  boundary_union = dissolved.geometry.boundary.unary_union


,new_sd,voters,pct
0,1,768579,0.482849
1,2,823180,0.517151
